# ДЗ1. Обучение ML-модели

Курсовой проект ML/DLOps. Анализ данных Inside Airbnb.

Цель работы: загрузить данные из S3-хранилища Yandex Cloud, подготовить признаки, обучить модель машинного обучения и оценить её качество на тестовой выборке.

### Импорт библиотек

In [1]:
!pip install -q boto3

import os
import boto3
import pandas as pd

### Настраиваем доступ к S3

In [2]:
# Получаем ключи из переменных окружения
if not os.getenv("AWS_ACCESS_KEY_ID") or not os.getenv("AWS_SECRET_ACCESS_KEY"):
    try:
        from google.colab import userdata

        os.environ["AWS_ACCESS_KEY_ID"] = userdata.get("AWS_ACCESS_KEY_ID")
        os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
    except ImportError:
        raise RuntimeError(
            "Необходимо задать AWS_ACCESS_KEY_ID и AWS_SECRET_ACCESS_KEY"
        )

S3_BUCKET = "mlops26-course-bucket"
S3_KEY = "prepared/listings.csv"

s3 = boto3.client(
    "s3",
    endpoint_url="https://storage.yandexcloud.net",
    region_name="ru-central1"
)

print("Клиент S3 создан")


Клиент S3 создан


### Читаем данные и смотрим таблицу

In [3]:
response = s3.get_object(
    Bucket=S3_BUCKET,
    Key=S3_KEY
)

with response["Body"] as body:
    df = pd.read_csv(body)

print("Размер таблицы:", df.shape)

display(df.head())

print("Названия столбцов:")
print(df.columns.tolist())

Размер таблицы: (20151, 13)


,listing_id,snapshot_date,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,neighbourhood_cleansed,property_type,room_type,price
0,2145993,2026-09-17,51.511100,-0.198250,2,1.0,1.0,1.0,4.0,Kensington and Chelsea,Entire rental unit,Entire home/apt,251.75
1,846381013872498937,2026-09-17,51.599540,-0.064500,6,1.5,3.0,4.0,2.0,Haringey,Entire home,Entire home/apt,201.60
2,1230990949123993577,2026-06-19,51.371535,-0.097551,2,1.0,NaN,1.0,1.0,Croydon,Private room in rental unit,Private room,73.00
3,54178450,2026-09-17,51.390740,-0.273480,2,1.0,1.0,1.0,1.0,Kingston upon Thames,Entire condo,Entire home/apt,126.00
4,1297347987655075325,2026-03-21,51.517398,0.012953,4,1.0,1.0,2.0,3.0,Newham,Entire rental unit,Entire home/apt,147.00


Названия столбцов:
['listing_id', 'snapshot_date', 'latitude', 'longitude', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'minimum_nights', 'neighbourhood_cleansed', 'property_type', 'room_type', 'price']


Буду решать задачу регрессии: прогнозирование стоимости аренды жилья Airbnb.

У нас есть числовая целевая переменная price и признаки, которые потенциально связаны со стоимостью жилья.

### Первичный анализ данных

In [4]:
# Типы данных и пропуски
info_df = pd.DataFrame({
    "Тип данных": df.dtypes,
    "Пропуски": df.isna().sum(),
    "Доля пропусков, %": (df.isna().mean() * 100).round(2)
})

display(info_df)

# Проверяем повторяющиеся объявления
print("Уникальных объявлений:", df["listing_id"].nunique())
print("Повторных listing_id:", df["listing_id"].duplicated().sum())

# Смотрим распределение цены
print("Статистика по цене:")
display(df["price"].describe())


,Тип данных,Пропуски,"Доля пропусков, %"
listing_id,int64,0,0.00
snapshot_date,object,0,0.00
latitude,float64,0,0.00
longitude,float64,0,0.00
accommodates,int64,0,0.00
bathrooms,float64,31,0.15
bedrooms,float64,1548,7.68
beds,float64,682,3.38
minimum_nights,float64,40,0.20
neighbourhood_cleansed,object,0,0.00


Уникальных объявлений: 8986
Повторных listing_id: 11165
Статистика по цене:


,price
count,20151.000000
mean,250.966359
std,484.703977
min,4.320000
25%,99.550000
50%,174.000000
75%,286.000000
max,23686.000000


В данных 20 151 строка и 13 столбцов. Больше всего пропусков в признаках `bedrooms` (7,68%) и `beds` (3,38%). При подготовке данных эти пропуски нужно будет обработать.

Также выяснилось, что в таблице только 8 986 уникальных объявлений, то есть одни и те же объекты встречаются несколько раз. Это важно учитывать при разделении данных на обучающую и тестовую выборки.

Цены имеют большой разброс: медиана составляет 174, среднее значение около 251, а максимальная цена достигает 23 686. Вероятно, дорогие объекты будут влиять на ошибки прогнозирования.

### Разделение на train и test

In [5]:
from sklearn.model_selection import GroupShuffleSplit

# Признаки для обучения
features = [
    "latitude",
    "longitude",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "minimum_nights",
    "neighbourhood_cleansed",
    "property_type",
    "room_type"
]

X = df[features].copy()
y = df["price"].copy()

# Разделяем данные по уникальным объявлениям
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=df["listing_id"])
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print("Размер train:", X_train.shape)
print("Размер test:", X_test.shape)

# Проверяем отсутствие одинаковых объявлений
train_ids = set(df.iloc[train_idx]["listing_id"])
test_ids = set(df.iloc[test_idx]["listing_id"])

print("Общих объявлений:", len(train_ids & test_ids))


Размер train: (16076, 10)
Размер test: (4075, 10)
Общих объявлений: 0


Данные разделены на обучающую и тестовую выборки в соотношении примерно 80/20. Для разделения использовался `GroupShuffleSplit` с группировкой по `listing_id`, поскольку одно объявление может встречаться в таблице несколько раз.

Проверка показала, что общих объявлений между выборками нет. Таким образом, модель будет проверяться на объектах, которые не использовались при обучении.

### Предобработка признаков

In [6]:

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

# Числовые признаки
numeric_features = [
    "latitude",
    "longitude",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "minimum_nights"
]

# Категориальные признаки
categorical_features = [
    "neighbourhood_cleansed",
    "property_type",
    "room_type"
]

# Заполняем пропуски в числовых признаках
numeric_transformer = SimpleImputer(strategy="median")

# Заполняем пропуски и кодируем категории
from sklearn.pipeline import make_pipeline

categorical_transformer = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore")
)

# Объединяем преобразования
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# Настраиваем преобразования только на train
X_train_processed = preprocessor.fit_transform(X_train)

# Применяем те же преобразования к test
X_test_processed = preprocessor.transform(X_test)

print("Размер train после обработки:", X_train_processed.shape)
print("Размер test после обработки:", X_test_processed.shape)


Размер train после обработки: (16076, 99)
Размер test после обработки: (4075, 99)


После предобработки количество признаков увеличилось с 10 до 99 за счёт кодирования категориальных переменных. Пропуски в числовых признаках были заполнены медианой, а в категориальных наиболее частыми значениями.

Преобразования настраивались только на обучающей выборке, после чего применялись к тестовой. Размеры выборок по количеству строк не изменились.

### Обучение модели Random Forest

In [7]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Создаём модель
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    random_state=42,
    n_jobs=-1
)

# Обучаем модель
model.fit(X_train_processed, y_train)

# Получаем предсказания на тестовой выборке
y_pred = model.predict(X_test_processed)

# Рассчитываем метрики качества
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2:", round(r2, 4))

MAE: 102.47
RMSE: 360.57
R2: 0.1592


### Сравнение с базовым прогнозом

In [8]:
from sklearn.dummy import DummyRegressor

# Базовая модель: всегда прогнозирует среднюю цену train
baseline = DummyRegressor(strategy="mean")

baseline.fit(X_train_processed, y_train)

y_pred_baseline = baseline.predict(X_test_processed)

# Метрики базовой модели
baseline_mae = mean_absolute_error(y_test, y_pred_baseline)
baseline_rmse = np.sqrt(mean_squared_error(y_test, y_pred_baseline))
baseline_r2 = r2_score(y_test, y_pred_baseline)

# Сравниваем результаты
results = pd.DataFrame({
    "Модель": ["Baseline", "Random Forest"],
    "MAE": [baseline_mae, mae],
    "RMSE": [baseline_rmse, rmse],
    "R2": [baseline_r2, r2]
})

display(results.round(4))


,Модель,MAE,RMSE,R2
0,Baseline,166.2629,393.2430,-0.0000
1,Random Forest,102.4676,360.5703,0.1592


**Вывод:** В работе была обучена модель Random Forest для прогнозирования стоимости жилья Airbnb. Данные были разделены на обучающую и тестовую выборки с учётом listing_id, чтобы одни и те же объявления не попадали в обе выборки.

На тестовой выборке модель показала MAE = 102,47, RMSE = 360,57 и R² = 0,1592. По сравнению с базовым прогнозом средней цены MAE снизилась примерно на 38,4%, а RMSE на 8,3%.

Таким образом, Random Forest показала результат лучше базовой модели, хотя качество прогноза остаётся невысоким. На ошибки могут влиять большой разброс цен, дорогие объекты и ограниченный набор признаков. В дальнейшем можно попробовать улучшить качество модели.